# distributed-sampler-shard — faded example 3: Construct DistributedSampler with the correct keyword arguments

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `distributed-sampler-shard`. The last cell reports your progress on the `Distributed: DistributedSampler shard` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: DistributedSampler shard` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`distributed-sampler-shard`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "distributed-sampler-shard"
DD_SUBTOPIC = "Distributed: DistributedSampler shard"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`DistributedSampler` takes four key arguments: `dataset` (the data source), `num_replicas` (total number of ranks in the process group), `rank` (this process's rank), and optionally `shuffle` and `seed`. The `num_replicas` argument controls how many equal-length shards are produced; each rank gets exactly one shard. Passing `shuffle=False` gives a deterministic round-robin split without randomization.

## Faded exercise 3

Complete `make_sampler`. Construct and return a `DistributedSampler` with `shuffle=False` (deterministic) for the given rank and world_size.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
from torch.utils.data.distributed import DistributedSampler

def make_sampler(dataset, world_size, rank):
    sampler = None  # TODO: fill in this step — read the prompt cell above
    return sampler


def _test():
    import torch
    from torch.utils.data import TensorDataset
    from itertools import chain

    n = 9
    world_size = 3
    dataset = TensorDataset(torch.arange(n))

    shards = [list(make_sampler(dataset, world_size, r)) for r in range(world_size)]

    # All shards same length
    assert all(len(s) == len(shards[0]) for s in shards)

    # Union covers full dataset
    assert set(chain(*shards)) == set(range(n))

    # Pairwise disjoint (n divisible by world_size)
    for i in range(world_size):
        for j in range(i + 1, world_size):
            assert len(set(shards[i]) & set(shards[j])) == 0

    # shuffle=False: deterministic, same result on repeated calls
    shards2 = [list(make_sampler(dataset, world_size, r)) for r in range(world_size)]
    assert shards == shards2


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from torch.utils.data.distributed import DistributedSampler

def make_sampler(dataset, world_size, rank):
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=False)
    return sampler
```
</details>